https://www.kaggle.com/c/bike-sharing-demand/data?select=train.csv

- 누수 체크가 1순위
- RMSLE 문제는 log 타깃 학습이 기본
- 베이스라인으로 빠르게 방향 결정(선형 vs 트리)
- 중요도 기반으로 FE(피처엔지니어링)를 선택(감이 아니라 근거)
- 시간 데이터는 검증을 시간기반으로 해야 실전 성능을 보장

# 데이터 수집

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


plt.rc('figure', figsize=(10, 6))

from matplotlib import rcParams
rcParams['font.family'] = 'New Gulim'
rcParams['font.size'] = 10
rcParams['axes.unicode_minus'] = False

In [7]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# log 값 변환 시 NaN등의 이슈로 log() 가 아닌 log1p() 를 이용하여 RMSLE 계산
def rmsle(y, pred):
    log_y = np.log1p(y)
    log_pred = np.log1p(pred)
    squared_error = (log_y - log_pred) ** 2
    rmsle = np.sqrt(np.mean(squared_error))
    return rmsle

# 사이킷런의 mean_squared_error() 를 이용하여 RMSE 계산
def rmse(y, pred):
    return np.sqrt(mean_squared_error(y, pred))

# RMSLE, RMSE, MAE 를 모두 계산
def evaluate_regr(y, pred):
    rmsle_val = rmsle(y, pred)
    rmse_val  = rmse(y, pred)
    mae_val   = mean_absolute_error(y, pred)
    print('RMSLE: {:.3f}, RMSE: {:.3f}, MAE: {:.3f}'.format(rmsle_val, rmse_val, mae_val))


- 회귀 모델의 성능평가에서 R2 보다는 MAE 를 활용한다.

In [8]:
# 데이터 로딩
bike_df = pd.read_csv('data/bike_train.csv')
print(bike_df.shape)
bike_df.head()

(10886, 12)


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


datetime	season	holiday	workingday	weather	temp	atemp	humidity	windspeed	casual	registered	count
0	2011-01-01 00:00:00	1	0	0	1	9.84	14.395	81	0.0	3	13	16
1	2011-01-01 01:00:00	1	0	0	1	9.02	13.635	80	0.0	8	32	40
2	2011-01-01 02:00:00	1	0	0	1	9.02	13.635	80	0.0	5	27	32

캐글 바이크데이터를 이용한 자전거대여량(COUNT)을 예측하는 회귀모델을 만듭니다.
평가 방법은 RMSLE, RMSE, MAE를 사용합니다.
전체 프로세스를 설계하고 내가 검토할 수 있도록 제시해 보세요.
나의 피드백을 반영해서 각 단계를 진행하는 것이 원칙입니다.
나의 개발환경은 IPYNB 파일이고, 각 코드의 의미있는 주석을 포함하세요.
코드의 실행결과를 내가 제시하면, 피드백(개선포인트)을 마크다운 형식으로 간결하고 명료한 표현으로 생성하세요.


# 데이터 탐색

In [9]:
import numpy as np
import pandas as pd

# 1단계) 데이터 구조 점검 (shape / head / columns / dtypes)

In [10]:
print(bike_df.columns.tolist())
print("shape:", bike_df.shape)
print(bike_df.dtypes)

['datetime', 'season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'casual', 'registered', 'count']
shape: (10886, 12)
datetime          str
season          int64
holiday         int64
workingday      int64
weather         int64
temp          float64
atemp         float64
humidity        int64
windspeed     float64
casual          int64
registered      int64
count           int64
dtype: object


컬럼: datetime 포함, casual, registered 포함, 타깃 count 포함

크기: (10886, 12)

타입: datetime이 str → datetime 변환 필요

중요: casual, registered는 누수 가능성 매우 큼(보통 casual + registered = count) → 피처에서 제외 예정

In [11]:
# [STEP 2] 누수(leakage) 확인 + 결측치 점검
# 1) casual + registered = count 관계 확인 (대부분 동일하면 누수 확정)
leak_equal_cnt = ((bike_df["casual"] + bike_df["registered"]) == bike_df["count"]).sum()
total = len(bike_df)
print(f"leak check: (casual+registered)==count  -> {leak_equal_cnt}/{total} ({leak_equal_cnt/total:.4%})")

# 2) 결측치 확인
print("\n[missing values]")
print(bike_df.isnull().sum().sort_values(ascending=False).head(20))

leak check: (casual+registered)==count  -> 10886/10886 (100.0000%)

[missing values]
datetime      0
season        0
holiday       0
workingday    0
weather       0
temp          0
atemp         0
humidity      0
windspeed     0
casual        0
registered    0
count         0
dtype: int64


# datetime 변환 + 시간 파생 피처 생성

In [12]:
bike_step3 = bike_df.copy()

# 1) datetime 문자열 -> datetime 타입 변환
bike_step3["datetime"] = pd.to_datetime(bike_step3["datetime"])
bike_step3.info()

<class 'pandas.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   datetime    10886 non-null  datetime64[us]
 1   season      10886 non-null  int64         
 2   holiday     10886 non-null  int64         
 3   workingday  10886 non-null  int64         
 4   weather     10886 non-null  int64         
 5   temp        10886 non-null  float64       
 6   atemp       10886 non-null  float64       
 7   humidity    10886 non-null  int64         
 8   windspeed   10886 non-null  float64       
 9   casual      10886 non-null  int64         
 10  registered  10886 non-null  int64         
 11  count       10886 non-null  int64         
dtypes: datetime64[us](1), float64(3), int64(8)
memory usage: 1020.7 KB


In [13]:
# 2) 파생 피처 생성
bike_step3["year"] = bike_step3["datetime"].dt.year
bike_step3["month"] = bike_step3["datetime"].dt.month
bike_step3["day"] = bike_step3["datetime"].dt.day
bike_step3["hour"] = bike_step3["datetime"].dt.hour
bike_step3["dayofweek"] = bike_step3["datetime"].dt.dayofweek  # 월=0, 요일

In [14]:
bike_step3['workingday'].value_counts()

workingday
1    7412
0    3474
Name: count, dtype: int64

In [15]:
bike_step3['holiday'].value_counts()

holiday
0    10575
1      311
Name: count, dtype: int64

# 파생변수 추가 : 주말/출퇴근시간대 (선택)

In [16]:
# 주말/출퇴근 시간대
# bike_step3["is_weekend"] = (bike_step3["dayofweek"] >= 5).astype(int)
# bike_step3["rush_hour"] = bike_step3["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)

In [17]:
# 체감-실제 온도 차이
# bike_step3["temp_diff"] = bike_step3["atemp"] - bike_step3["temp"]

In [18]:
bike_step3.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,year,month,day,hour,dayofweek
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16,2011,1,1,0,5
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40,2011,1,1,1,5
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32,2011,1,1,2,5
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13,2011,1,1,3,5
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1,2011,1,1,4,5


# 학습용 데이터 구성(X/y) + 홀드아웃 분리

In [19]:
from sklearn.model_selection import train_test_split

bike_step4 = bike_step3.copy()

# 1) 누수 컬럼 및 원본 datetime 제거
drop_cols = ["datetime", "casual", "registered"]
X = bike_step4.drop(columns=["count"] + drop_cols)
y = bike_step4["count"].copy()

# 2) 홀드아웃 분리
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_train shape:", y_train.shape)
print("y_valid shape:", y_valid.shape)

print("\n[train columns]")
print(X_train.columns.tolist())

X_train shape: (8708, 13)
X_valid shape: (2178, 13)
y_train shape: (8708,)
y_valid shape: (2178,)

[train columns]
['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'year', 'month', 'day', 'hour', 'dayofweek']


In [20]:
X_train.head()

,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,year,month,day,hour,dayofweek
2815,3,0,1,1,27.88,31.820,83,6.0032,2011,7,6,5,2
8695,3,0,0,1,36.90,40.910,39,19.9995,2012,8,4,16,5
8406,3,0,1,1,32.80,35.605,41,16.9979,2012,7,11,15,2
1543,2,0,0,2,14.76,18.180,93,7.0015,2011,4,10,4,6
4952,4,0,0,1,13.12,15.150,45,16.9979,2011,11,19,10,5


# 전처리(원-핫, 스케일러)

In [21]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# 1) 컬럼 타입 구분
categorical_cols = ["season","holiday","workingday","weather","year","month","day","hour","dayofweek"]
numeric_cols = ["temp","atemp","humidity","windspeed"]

# 2) 전처리: 범주형 원핫 + 수치형 스케일
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols),
    ],
    remainder="drop"
)


In [22]:
preprocess

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

# (전처리 → 모델) 파이프라인 구조
모델만 바꿔 끼우면서도 동일한 전처리/검증 규칙을 유지 -> 공정한 검증

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LinearRegression

In [24]:
# 공통: log1p 타깃
y_train_log = np.log1p(y_train)

# ---- (A) LinearRegression ----
lin = Pipeline(steps=[
    ("prep", preprocess),
    ("model", LinearRegression())
])
lin.fit(X_train, y_train_log)
pred_lin = np.expm1(lin.predict(X_valid))
pred_lin = np.maximum(0, pred_lin)  # rmsle 안전, log1p(pred)를 쓰기 때문에 pred가 음수 NaN
evaluate_regr(y_valid, pred_lin)

# ---- (B) Ridge ----
ridge = Pipeline(steps=[
    ("prep", preprocess),
    ("model", Ridge(alpha=5.0, random_state=42))
])
ridge.fit(X_train, y_train_log)
pred_ridge = np.expm1(ridge.predict(X_valid))
pred_ridge = np.maximum(0, pred_ridge)
evaluate_regr(y_valid, pred_ridge)

RMSLE: 0.581, RMSE: 95.190, MAE: 61.744
RMSLE: 0.582, RMSE: 95.679, MAE: 61.906


## Linear vs Ridge 결과 해석

- **LinearRegression**
  - RMSLE: **0.581**, RMSE: **95.190**, MAE: **61.744**
- **Ridge**
  - RMSLE: **0.582**, RMSE: **95.679**, MAE: **61.906**

### 결론
- 현재 홀드아웃(80/20, random_state=42) 기준으로는 **Linear가 근소하게 우세**합니다.
- 차이는 매우 작아서(3개 지표 모두 미세) “Ridge가 확실히 낫다/나쁘다”라기보다는,
  - **alpha(규제 강도)가 현재 값(아마 5.0)에서 최적이 아닐 가능성**
  - 또는 **규제가 거의 필요 없는 정도로 선형모델이 안정적인 데이터/전처리 상태**
  둘 중 하나로 보는 게 자연스럽습니다.

### 다음 개선 포인트 (가성비 높은 순)
1) **Ridge alpha만 짧게 스윕**해서 “진짜로 규제가 도움이 되는지” 확인  
   - 보통 `alpha`에 따라 RMSLE가 의미 있게 바뀌는 경우가 많습니다.
2) alpha 스윕 후에도 개선이 미미하면  
   - 선형계열(Linear/Ridge/Lasso/ElasticNet)로 큰 점수 향상은 어렵고
   - 다음 단계는 **비선형 모델(트리 부스팅)**로 넘어갈 타이밍입니다.

In [25]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
import numpy as np

alphas = [0.1, 1, 5, 10, 50]

for a in alphas:
    print(f"\n[Ridge alpha={a}]")
    ridge = Pipeline(steps=[
        ("prep", preprocess),
        ("model", Ridge(alpha=a, random_state=42))
    ])
    ridge.fit(X_train, np.log1p(y_train))
    pred = np.expm1(ridge.predict(X_valid))
    pred = np.maximum(0, pred)          
    evaluate_regr(y_valid, pred)        


[Ridge alpha=0.1]
RMSLE: 0.581, RMSE: 95.194, MAE: 61.745

[Ridge alpha=1]
RMSLE: 0.581, RMSE: 95.276, MAE: 61.770

[Ridge alpha=5]
RMSLE: 0.582, RMSE: 95.679, MAE: 61.906

[Ridge alpha=10]
RMSLE: 0.583, RMSE: 96.258, MAE: 62.123

[Ridge alpha=50]
RMSLE: 0.606, RMSE: 102.159, MAE: 64.880


## STEP 7 Ridge alpha 스윕 피드백

### 결과 요약
- alpha=0.1 → **RMSLE 0.581**, RMSE 95.194, MAE 61.745  ✅(베스트 그룹)
- alpha=1   → **RMSLE 0.581**, RMSE 95.276, MAE 61.770  ✅(베스트 그룹)
- alpha=5   → RMSLE 0.582, RMSE 95.679, MAE 61.906
- alpha=10  → RMSLE 0.583, RMSE 96.258, MAE 62.123
- alpha=50  → RMSLE 0.606, RMSE 102.159, MAE 64.880  ❌(과규제)

### 해석
- 규제가 강해질수록(alpha↑) 모든 지표가 **일관되게 악화**됩니다.
- 즉, 현재 피처/원핫 구조에서는 **선형모델이 이미 충분히 안정적**이라 강한 규제가 도움되지 않습니다.
- Ridge의 최적은 사실상 **alpha ≈ 0.1~1** 영역이며, 성능은 Linear와 **동급(거의 동일)**입니다.

### 다음 액션 추천
- 선형계열(Linear/Ridge)로는 큰 개선 폭이 제한적이니,
  다음 단계는 **비선형 모델로 넘어가는 게 가성비가 좋습니다.**
- 우선순위: `HistGradientBoostingRegressor` (빠르고 강력한 기본 트리 부스팅)

# HistGradientBoostingRegressor
전처리는 원-핫 + 수치 그대로로 두고, 스케일링은 제외 (트리구조 계열은 불필요)
HistGradientBoostingRegressor`는 **입력 X가 dense(밀집 배열)** 이어야 한다
- sparse_output=False

In [26]:
from sklearn.ensemble import HistGradientBoostingRegressor

In [27]:
# 트리 모델용 전처리: 범주형 원-핫, 수치형은 그대로 통과
preprocess_hgb = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(sparse_output=False,handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ],
    remainder="drop"
)

hgb = Pipeline(steps=[
    ("prep", preprocess_hgb),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=500,
        random_state=42
    ))
])

hgb.fit(X_train, np.log1p(y_train))
pred = np.expm1(hgb.predict(X_valid))
pred = np.maximum(0, pred)
evaluate_regr(y_valid, pred)

RMSLE: 0.298, RMSE: 43.344, MAE: 25.894


## STEP 8 (HGB) 결과 피드백

### 성능 비교 (Linear 대비)
- Linear(기준): **RMSLE 0.581 / RMSE 95.190 / MAE 61.744**
- HGB: **RMSLE 0.298 / RMSE 43.344 / MAE 25.894**

➡️ **압도적 개선**입니다.  
- RMSLE가 **0.581 → 0.298**로 크게 감소(상대적으로 거의 절반 수준)
- RMSE/MAE도 **절반 이상 감소**  
→ 현재 데이터/피처에서는 **비선형(트리 부스팅) 모델이 정답에 가깝게 맞습니다.**

### 해석 & 개선 포인트
1) **모델 선택 방향 확정**
- 이제 베이스라인은 선형이 아니라 **HGB를 기준 모델**로 잡는 게 합리적입니다.

2) **다음 튜닝 우선순위(가성비)**
- `max_depth`(복잡도)와 `learning_rate ↔ max_iter`(학습 강도) 조절이 가장 영향 큽니다.
- 현재 설정도 이미 매우 좋지만, 과적합/일반화 확인을 위해 “한두 개 조합”만 더 보는 게 좋습니다.

3) **검증 전략 강화 필요**
- 랜덤 홀드아웃에서 성능이 크게 좋아진 만큼,
  다음은 **시간 기반 검증(TimeSeriesSplit 또는 날짜 기준 후반부 validation)**로
  “미래 예측 성능”이 유지되는지 확인하는 단계가 필요합니다.

In [28]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
import numpy as np

print("[RFR]")
rfr = Pipeline(steps=[
    ("prep", preprocess),  # 기존 preprocess(원-핫 + 스케일) 그대로 사용
    ("model", RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

rfr.fit(X_train, np.log1p(y_train))
pred = np.expm1(rfr.predict(X_valid))
pred = np.maximum(0, pred)
evaluate_regr(y_valid, pred)

[RFR]
RMSLE: 0.330, RMSE: 45.456, MAE: 27.590


## HGB vs RFR 비교 피드백

### 결과 요약
- **HGB**: RMSLE **0.298**, RMSE 43.344, MAE 25.894
- **RFR**: RMSLE **0.330**, RMSE 45.456, MAE 27.590

➡️ **HGB가 3개 지표 모두 우세**합니다.  
- RMSLE 기준으로도 HGB가 확실히 낮고(0.298 < 0.330),
- RMSE/MAE도 함께 더 좋습니다.

### 해석
- 이 데이터(시간대+날씨 기반)는 **부스팅 계열이 랜덤포레스트보다 더 잘 맞는 패턴**이 흔합니다.  
  부스팅은 “잔차를 순차적으로 보정”해서 복잡한 비선형을 더 세밀하게 잡는 경향이 있어요.
- RFR은 튜닝을 해도 HGB 수준까지 따라붙을 때가 있지만, 대개 **깊이/leaf 제한을 잘 걸어야** 하고, 그래도 부스팅이 앞서는 경우가 많습니다.

### 다음 개선 포인트 (한 스텝 후보)
1) **RFR 튜닝으로 따라붙을 여지 확인**
   - `min_samples_leaf`를 키우면 일반화가 좋아지는 경우가 많아요.
   - 추천 1차: `min_samples_leaf=5` 또는 `10`, 그리고 `max_features="sqrt"`  
2) **HGB 튜닝으로 추가 개선 노리기 (더 유망)**
   - `max_depth`(6/8/10) 또는 `learning_rate/max_iter` 조합 2~3개만 비교
3) **검증 신뢰도 강화**
   - 랜덤 홀드아웃에서 크게 좋아진 모델은 **시간기반 검증**으로 재확인이 중요합니다.

# XGB, LGBM 추가 비교 

일반적으로 부스팅 lgbm 알고리즘이 성능이 잘 나옴 > 추가 요구함.

## train_and_evaluate() 정의
모델별 학습/예측/평가 파이프라인 실행 함수

In [29]:
# 모델 1개 학습/예측/평가를 동일 규칙으로 수행하는 함수 추가

import numpy as np
from sklearn.pipeline import Pipeline

def train_and_evaluate(model, name, preprocess_used):
    """
    - model: sklearn 호환 regressor
    - preprocess_used: 이미 만들어 둔 전처리기(preprocess 또는 preprocess_hgb 등)
    """
    pipe = Pipeline([
        ("prep", preprocess_used),
        ("model", model)
    ])
    pipe.fit(X_train, np.log1p(y_train))
    pred = np.expm1(pipe.predict(X_valid))
    pred = np.maximum(0, pred)  # rmsle(log1p) 보호 + count 도메인 반영
    print(f"\n[{name}]")
    evaluate_regr(y_valid, pred)

## XGBRegressor

In [30]:
# 1) XGBRegressor
try:
    from xgboost import XGBRegressor

    xgb = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )
    train_and_evaluate(xgb, "XGBRegressor", preprocess)

except Exception as e:
    print("[XGBRegressor] 실행 실패:", type(e).__name__, str(e)[:200])



[XGBRegressor]
RMSLE: 0.289, RMSE: 39.047, MAE: 23.798


## XGB 결과 피드백 (HGB/RFR와 비교)

### 결과 요약
- **HGB**: RMSLE 0.298 / RMSE 43.344 / MAE 25.894
- **RFR**: RMSLE 0.330 / RMSE 45.456 / MAE 27.590
- **XGB**: **RMSLE 0.289 / RMSE 39.047 / MAE 23.798** ✅

### 해석
- **XGB가 현재 1등**입니다.
  - RMSLE: 0.298 → **0.289** (개선)
  - RMSE/MAE도 함께 개선되어, 단순히 로그스케일만 좋아진 게 아니라 **절대 오차도 줄었습니다.**
- 즉, 지금 데이터/피처에서는 “선형 < RF < HGB < XGB” 흐름이 명확합니다.

### 다음 개선 포인트
1) **LGBM 추가 비교**
- 보통 이 문제에서 LGBM이 XGB보다 더 좋거나 비슷하게 나오는 케이스가 많아서(튜닝 전이라도),
  다음 한 스텝으로 LGBM을 동일 조건으로 돌려보는 게 합리적입니다.

2) **검증 신뢰도**
- 성능이 좋아진 만큼, 이후에는 시간기반 검증(예: 뒤쪽 기간을 통째로 valid로)로
  “미래 예측”에서도 유지되는지 확인하면 선택이 더 확실해집니다.

## LGBMRegressor

In [31]:
# [LGBMRegressor] 1회 비교 실행
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.02,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

train_and_evaluate(lgbm, "LGBMRegressor", preprocess)


[LGBMRegressor]
RMSLE: 0.296, RMSE: 40.762, MAE: 24.512


c:\Users\Admin\miniconda3\envs\ml_edu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## LGBM 결과 피드백 (XGB/HGB/RFR와 비교)

### 결과 요약
- **XGBRegressor**: ✅ RMSLE **0.289**, RMSE 39.047, MAE 23.798  (현재 1등)
- **LGBMRegressor**: RMSLE **0.296**, RMSE 40.762, MAE 24.512
- **HGB**: RMSLE 0.298, RMSE 43.344, MAE 25.894
- **RFR**: RMSLE 0.330, RMSE 45.456, MAE 27.590
- **Linear**: RMSLE ~0.581 (baseline)

### 해석
- LGBM은 HGB보다 약간 좋지만, **XGB가 모든 지표에서 더 좋습니다.**
- 현재 설정(학습률/트리수/깊이/리프수)이 XGB에 더 잘 맞았거나,
  LGBM은 같은 노력 대비 **파라미터(특히 num_leaves/min_data_in_leaf)** 조정 여지가 남아있는 상태로 보입니다.

### 다음 액션 추천 (한 스텝)
- 이제 “모델 선택”은 **XGB로 확정**해도 되는 근거가 충분합니다(현재 검증 기준 1등).
- 다음 한 스텝은 둘 중 하나가 가장 효율적입니다:
  1) **XGB 소규모 튜닝(3~5개 조합)**: max_depth / min_child_weight / subsample / colsample_bytree
  2) **시간기반 검증 전환**: 랜덤 홀드아웃이 아니라 날짜 후반부를 valid로 두고 “미래 예측 성능” 확인

### 추천 액션의 재 검토 요청 결과 추천 전략(실무/캐글에서 흔한 흐름)

1) 중요도 분석으로 “먹히는 축”을 파악

2) 가장 영향 큰 축에 FE 2~3개만 추가

3) 그 다음에 XGB 튜닝 (또는 튜닝→FE 순서도 가능하지만, 지금은 FE 여지가 커 보임)


# xgb 기반 피처 중요도 분석

In [32]:
xgb_fixed  = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

pipe_xgb = Pipeline([
    ("prep", preprocess),
    ("model", xgb_fixed )
])

pipe_xgb.fit(X_train, np.log1p(y_train))


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

In [33]:
# 2) 원-핫 포함 최종 feature name 추출
feat_names = pipe_xgb.named_steps["prep"].get_feature_names_out()

# 3) XGB gain 기반 feature importance (더미 피처 단위)
importances = pipe_xgb.named_steps["model"].feature_importances_

imp_df = pd.DataFrame({"feature": feat_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False)

display(imp_df.head(30))

,feature,importance
49,cat__hour_4,0.166008
48,cat__hour_3,0.155281
50,cat__hour_5,0.106497
47,cat__hour_2,0.102616
46,cat__hour_1,0.083798
45,cat__hour_0,0.042620
7,cat__workingday_1,0.035928
62,cat__hour_17,0.027488
63,cat__hour_18,0.024934
6,cat__workingday_0,0.023391


## Feature Importance 기반 인사이트 (XGB)

### 상위 중요도 특징
- **hour 더미가 압도적**: `hour_0~6`(특히 3~5시)와 `hour_17~19`가 상위권을 거의 독점
- 그 다음 축: **workingday(근무일 여부)**, **year**, **season**, **weather**
- 수치형은 `temp/atemp`가 상위에 보이지만, hour에 비해 비중이 작음
- `dayofweek`는 주말(5,6)이 상대적으로 의미 있음(하지만 hour보단 약함)

### 해석
- 현재 모델은 “시간대별 기본 수요 패턴”을 거의 핵심 신호로 사용 중입니다.
- 그런데 hour를 원-핫으로만 넣으면, **시간의 연속성/주기성(24시간 순환)**을 직접 표현하지 못합니다.
  → 이 지점이 **피처 엔지니어링으로 점수를 더 깎을 수 있는 가장 유력한 구간**입니다.

엔지니어링 추가로 “튜닝 대비” 더 좋아질 가능성이 큰 피처들 (우선순위)
1) 시간 주기성(sin/cos) 피처 (강추)

왜? hour가 가장 중요하지만 원-핫은 “0시와 23시가 멀다”라고 인식합니다.
sin/cos는 24시간 원형 주기를 직접 반영합니다.

hour_sin = sin(2π*hour/24)

hour_cos = cos(2π*hour/24)

(옵션) month_sin/cos도 계절성을 부드럽게 반영

2) 러시아워/주말 피처 (당신이 보류했지만, 중요도 보면 재검토 가치 큼)

중요도에서 hour_17,18,19 + workingday가 높아서,

rush_hour (7~9, 17~19)

is_weekend (dayofweek 5,6)

그리고 상호작용: rush_hour * workingday (근무일 출퇴근 효과)

3) 날씨 상호작용/비선형화

humidity * temp, windspeed * temp 같은 곱

weather가 3까지 중요도가 보이므로 weather와 온도/습도의 상호작용도 후보

## sin/cos 2개만 추가

In [34]:
import numpy as np

# X_train/X_valid에 hour_sin/cos 추가 (원본을 건드리지 않기 위해 copy)
X_train2 = X_train.copy()
X_valid2 = X_valid.copy()

X_train2["hour_sin"] = np.sin(2*np.pi*X_train2["hour"]/24)
X_train2["hour_cos"] = np.cos(2*np.pi*X_train2["hour"]/24)
X_valid2["hour_sin"] = np.sin(2*np.pi*X_valid2["hour"]/24)
X_valid2["hour_cos"] = np.cos(2*np.pi*X_valid2["hour"]/24)

# numeric_cols에 새 피처 반영 (전역 변수면 append)
numeric_cols2 = numeric_cols + ["hour_sin", "hour_cos"]

# preprocess를 hour_sin/cos 포함 버전으로 다시 정의
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess2 = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols2),
    ],
    remainder="drop"
)

XGB를 동일 파라미터로 다시 1번

In [35]:
import numpy as np
from sklearn.pipeline import Pipeline

def fit_eval_xgb(preprocess_used, X_tr, X_va, label="XGB"):
    pipe = Pipeline([("prep", preprocess_used), ("model", xgb_fixed)])
    pipe.fit(X_tr, np.log1p(y_train))
    pred = np.maximum(0, np.expm1(pipe.predict(X_va)))
    print(f"\n[{label}]")
    evaluate_regr(y_valid, pred)

In [36]:
fit_eval_xgb(preprocess2, X_train2, X_valid2, label="XGB + hour/sin/cos")


[XGB + hour/sin/cos]
RMSLE: 0.271, RMSE: 34.750, MAE: 20.924


## Feature Engineering 결과 피드백 (hour_sin/cos 추가 효과)

### 성능 변화 (XGB 기준)
- 기존 XGB: **RMSLE 0.289 / RMSE 39.047 / MAE 23.798**
- + `hour_sin`, `hour_cos`: ✅ **RMSLE 0.271 / RMSE 34.750 / MAE 20.924**

➡️ **큰 폭의 개선**입니다.
- RMSLE가 **0.018p 감소**(0.289 → 0.271)
- RMSE/MAE도 함께 크게 감소 → “로그 스케일만 좋아진 것”이 아니라 **실제 오차도 줄어듦**
- 결론: 이 문제에서는 **하이퍼파라미터 튜닝보다 FE(주기성 반영)가 더 강력한 레버**였습니다.

### 왜 좋아졌나 (중요도와 연결)
- 중요도에서 hour 더미가 압도적이었는데,
- 원-핫만으로는 **시간의 연속성과 24시간 주기성**을 모델이 직접 표현하기 어려움
- `sin/cos`가 그 구조를 제공하면서, 트리가 더 적은 분기로도 패턴을 잡아 일반화가 좋아진 것으로 해석됩니다.

month_sin/cos 추가

In [37]:
# 1) 기존(hour_sin/cos 추가된) X_train2/X_valid2를 기반으로 month_sin/cos 추가
X_train3 = X_train2.copy()
X_valid3 = X_valid2.copy()

X_train3["month_sin"] = np.sin(2*np.pi*X_train3["month"]/12)
X_train3["month_cos"] = np.cos(2*np.pi*X_train3["month"]/12)
X_valid3["month_sin"] = np.sin(2*np.pi*X_valid3["month"]/12)
X_valid3["month_cos"] = np.cos(2*np.pi*X_valid3["month"]/12)

# 2) numeric cols 확장
numeric_cols3 = numeric_cols2 + ["month_sin", "month_cos"]

# 3) 전처리 재정의
preprocess3 = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols3),
    ],
    remainder="drop"
)

In [38]:
fit_eval_xgb(preprocess3, X_train3, X_valid3, label="XGB + month sin/cos")


[XGB + month sin/cos]
RMSLE: 0.270, RMSE: 34.556, MAE: 20.864


## month_sin/cos 추가 결과 피드백

### 성능 변화 (XGB + hour_sin/cos 기준 대비)
- 기존( + hour_sin/cos ): **RMSLE 0.271 / RMSE 34.750 / MAE 20.924**
- 추가( + month_sin/cos ): ✅ **RMSLE 0.270 / RMSE 34.556 / MAE 20.864**

➡️ **개선은 있지만 폭은 작습니다.**
- RMSLE: 0.271 → 0.270 (미세 개선)
- RMSE/MAE도 같이 소폭 개선 → “의미 있는 방향”이긴 하나, 다음 큰 레버는 다른 쪽일 가능성이 큼

### 해석
- 월(month)은 이미 `season`/`weather`/`temp` 등과 중복 설명되는 부분이 많아서,
  주기 피처의 추가 이득이 hour_sin/cos만큼 크지 않은 게 자연스럽습니다.
- 그래도 “거의 공짜(피처 2개)”로 얻은 개선이라 유지 가치가 있습니다.

rush_hour + is_weekend + 상호작용(근무일)

In [39]:
# 1) 기존 X_train3/X_valid3(=hour_sin/cos + month_sin/cos 포함)에서 추가 피처 생성
X_train4 = X_train3.copy()
X_valid4 = X_valid3.copy()

# rush_hour: 7~9, 17~19
X_train4["rush_hour"] = X_train4["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)
X_valid4["rush_hour"] = X_valid4["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)

# is_weekend: 토(5), 일(6)
X_train4["is_weekend"] = (X_train4["dayofweek"] >= 5).astype(int)
X_valid4["is_weekend"] = (X_valid4["dayofweek"] >= 5).astype(int)

# 상호작용: 근무일 출퇴근 효과
X_train4["rush_work"] = X_train4["rush_hour"] * X_train4["workingday"]
X_valid4["rush_work"] = X_valid4["rush_hour"] * X_valid4["workingday"]

# 2) numeric cols 확장 (0/1 피처도 수치로 처리)
numeric_cols4 = numeric_cols3 + ["rush_hour", "is_weekend", "rush_work"]

# 3) 전처리 재정의
preprocess4 = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols4),
    ],
    remainder="drop"
)

In [40]:
fit_eval_xgb(preprocess4, X_train4, X_valid4, label="XGB + rush/weekend")


[XGB + rush/weekend]
RMSLE: 0.270, RMSE: 34.290, MAE: 20.814


## rush/weekend 피처 추가 결과 피드백

### 성능 비교 (직전 단계 대비)
- 직전(+ hour_sin/cos + month_sin/cos): **RMSLE 0.270 / RMSE 34.556 / MAE 20.864**
- 이번(+ rush_hour + is_weekend + rush_work): ✅ **RMSLE 0.270 / RMSE 34.290 / MAE 20.814**

### 해석
- **RMSLE는 동일(0.270)** → 로그 스케일 기준으로는 개선이 거의 없음
- **RMSE/MAE는 소폭 개선** → 절대 오차 측면에서는 약간 더 좋아짐
- 결론: 이 3개 피처는
  - “RMSLE 최적화” 관점에서는 영향이 크지 않지만,
  - “전체 오차(RMSE/MAE)까지 균형” 관점에서는 **유지 가치가 있는 편**입니다.

### 추천
- 현재 목표가 RMSLE 중심(캐글)이라면: **유지/보류 둘 다 가능** (효과가 미세)
- 다만 RMSE/MAE도 같이 보고 있고, 피처 자체가 해석 가능하므로: **유지 추천**

# 미래 데이터 기준 검증 > XGB 성능 유지 검토

GridSearchCV “랜덤 분할”이 아니라 시간순 정렬 후 마지막 20% 기간을 validation으로 두고, 지금까지 만든 FE(+hour_sin/cos, +month_sin/cos, +rush/weekend)를 그대로 적용해 XGB 성능이 유지되는지 확인

In [41]:
import numpy as np
import pandas as pd

# 1) 기존과 동일한 방식으로 "최종 FE 형태"의 전체 X를 만든다 (X_train4/X_valid4와 동일 컬럼 구성)
df = bike_df.copy()
df["datetime"] = pd.to_datetime(df["datetime"])

df["year"] = df["datetime"].dt.year
df["month"] = df["datetime"].dt.month
df["day"] = df["datetime"].dt.day
df["hour"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek

# 주기 FE
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

# rush/weekend FE
df["rush_hour"] = df["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
df["rush_work"] = df["rush_hour"] * df["workingday"]

# 누수 제거 + datetime 제거
drop_cols = ["datetime", "casual", "registered"]
X_full4 = df.drop(columns=["count"] + drop_cols)
y_full = df["count"].copy()

# 2) 시간기반 split 인덱스만 만든다 (마지막 20%)
order = df["datetime"].sort_values().index
cut = int(len(order) * 0.8)
train_idx = order[:cut]
valid_idx = order[cut:]

# 3) X_train4/X_valid4 형태 유지한 채로 분할한 데이터 생성
X_train4_ts = X_full4.loc[train_idx]
X_valid4_ts = X_full4.loc[valid_idx]
y_train_ts = y_full.loc[train_idx]
y_valid_ts = y_full.loc[valid_idx]

print("train period:", df.loc[train_idx, "datetime"].min(), "->", df.loc[train_idx, "datetime"].max())
print("valid period:", df.loc[valid_idx, "datetime"].min(), "->", df.loc[valid_idx, "datetime"].max())
print("shapes:", X_train4_ts.shape, X_valid4_ts.shape)

train period: 2011-01-01 00:00:00 -> 2012-08-05 04:00:00
valid period: 2012-08-05 05:00:00 -> 2012-12-19 23:00:00
shapes: (8708, 20) (2178, 20)


In [42]:
# fit_eval_xgb가 참조하는 y_train/y_valid를 시간기반 것으로 교체
y_train = y_train_ts
y_valid = y_valid_ts

# preprocess4는 현재 컬럼 구성과 일치해야 함
fit_eval_xgb(preprocess4, X_train4_ts, X_valid4_ts, label="XGB + rush/weekend (time split)")


[XGB + rush/weekend (time split)]
RMSLE: 0.340, RMSE: 73.488, MAE: 49.223


## 시간기반 검증 결과 피드백 (중요)

### 결과
- 랜덤 홀드아웃(최신 FE+XGB): **RMSLE ~0.270**
- 시간기반 split(마지막 20% 미래 구간): ❗ **RMSLE 0.340 / RMSE 73.488 / MAE 49.223**

➡️ **큰 폭으로 악화**되었습니다.  
이건 “모델이 갑자기 못해졌다”라기보다, **랜덤 분할이 미래 일반화 성능을 과대평가**했을 가능성이 큽니다.

### 왜 이런 일이 생기나 (가능성이 큰 원인 3가지)
1) **시간 분포 shift**
- 후반 기간(검증 구간)이 초반과 패턴/수요 수준이 달라서, 랜덤 분할처럼 섞어서 평가하면 쉽게 맞춰 보이지만
  실제 미래 예측에서는 성능이 떨어질 수 있습니다.

2) **'day' 피처의 일반화 문제**
- `day(1~31)`를 원-핫/범주처럼 쓰면, 특정 “일자 패턴”을 외우기 쉬운데
  미래 기간에서는 그 조합이 달라져 성능이 무너질 수 있습니다.
- 특히 month가 바뀌면 같은 day=1이라도 의미가 달라져요.

3) **검증 설계가 더 엄격해짐**
- 시간기반은 실제 배치 상황에 더 가까워서 RMSLE가 올라가는 게 자연스러운 면도 있습니다.

In [43]:
## day 제거” 실험

In [44]:
# day 제거 버전 전처리 만들기 (time-split 데이터는 그대로 재사용)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_cols_noday = ["season","holiday","workingday","weather","year","month","hour","dayofweek"]  # day 제거
numeric_cols_noday = numeric_cols4  # 이미 만든 수치 피처들(hour_sin/cos 등)은 그대로

preprocess4_noday = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols_noday),
        ("num", StandardScaler(), numeric_cols_noday),
    ],
    remainder="drop"
)

fit_eval_xgb(preprocess4_noday, X_train4_ts, X_valid4_ts, label="XGB (time split) - drop day")


[XGB (time split) - drop day]
RMSLE: 0.343, RMSE: 73.939, MAE: 48.920


# XGB 규제 강화로 time split 일반화 개선

time split에서 악화가 큰 경우, 보통 효과가 빠른 조합은 아래입니다.

max_depth 낮추기 (과적합 감소)

min_child_weight 올리기 (잎 노드 분할을 더 보수적으로)

subsample, colsample_bytree 낮추기 (랜덤성↑로 일반화↑)

(옵션) reg_alpha(L1) 추가, reg_lambda 증가

In [46]:
from xgboost import XGBRegressor

# 규제 강화 버전 (time split 일반화 목적)
xgb_fixed = XGBRegressor(
    n_estimators=4000,
    learning_rate=0.02,
    max_depth=6,
    min_child_weight=5,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    reg_alpha=0.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

fit_eval_xgb(preprocess4, X_train4_ts, X_valid4_ts, label="XGB (time split) + stronger regularization")


[XGB (time split) + stronger regularization]
RMSLE: 0.329, RMSE: 70.037, MAE: 45.796


## time split 구간 진단 (왜 성능 격차가 컸나)

### 분할 구간
- **train**: 2011-01-01 00:00:00 → 2012-08-05 04:00:00
- **valid**: 2012-08-05 05:00:00 → 2012-12-19 23:00:00
- shape: train (8708, 20) / valid (2178, 20)

### 해석 포인트
- valid가 **2012년 8월~12월(가을~초겨울)** 구간에 집중되어 있습니다.
- 자전거 수요는 계절/기온 변화에 민감하므로, 이 구간은 train 전체 평균과 분포가 달라질 가능성이 큽니다.
- 랜덤 홀드아웃은 이 계절 분포가 섞이기 때문에 “쉽게 맞춘 것처럼” 보일 수 있고,
  time split은 실제처럼 **계절 이동(shift)**을 그대로 맞아 더 어렵습니다.
- 그래서 규제 강화가 효과적이었던 것도 일관됩니다(특정 구간 패턴을 과도하게 외우지 않게 됨).

## min_child_weight만 2개 비교

In [47]:
from xgboost import XGBRegressor

for mcw in [2, 10]:
    xgb_fixed = XGBRegressor(
        n_estimators=4000,
        learning_rate=0.02,
        max_depth=6,
        min_child_weight=mcw,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_lambda=2.0,
        reg_alpha=0.0,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )
    fit_eval_xgb(preprocess4, X_train4_ts, X_valid4_ts, label=f"XGB time split (mcw={mcw})")


[XGB time split (mcw=2)]
RMSLE: 0.331, RMSE: 70.595, MAE: 46.406

[XGB time split (mcw=10)]
RMSLE: 0.329, RMSE: 69.169, MAE: 45.472


## min_child_weight 비교 피드백 (time split)

### 결과
- mcw=2  → RMSLE 0.331 / RMSE 70.595 / MAE 46.406
- ✅ mcw=10 → **RMSLE 0.329 / RMSE 69.169 / MAE 45.472**

### 해석
- `min_child_weight`를 올린(더 보수적인) 쪽이 **모든 지표에서 더 좋습니다.**
- 즉, time split(미래 구간)에서는 아직도 과적합 성향이 남아 있었고,
  더 강한 보수성(leaf 분할 조건 강화)이 일반화를 개선했습니다.

### 추천 설정(현재까지 time split 기준 베스트)
- max_depth=6
- min_child_weight=10
- subsample=0.7
- colsample_bytree=0.7
- learning_rate=0.02, n_estimators=4000
- reg_lambda=2.0

# 수치형 공선성 점검 + temp/atemp 선택

In [48]:
import pandas as pd
import numpy as np

# time split에서 사용 중인 최종 입력(X_full4)을 기준으로 수치형만 상관 확인
num_cols_check = ["temp","atemp","humidity","windspeed",
                  "hour_sin","hour_cos","month_sin","month_cos",
                  "rush_hour","is_weekend","rush_work"]

corr = X_full4[num_cols_check].corr().abs()

# 상삼각만 뽑아서 높은 상관 순으로 정렬
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper.stack().sort_values(ascending=False)

print(pairs.head(15))

temp        atemp        0.984948
rush_hour   rush_work    0.784820
temp        month_cos    0.690756
atemp       month_cos    0.678471
temp        month_sin    0.513653
atemp       month_sin    0.499116
humidity    hour_sin     0.367737
            windspeed    0.318607
is_weekend  rush_work    0.290795
humidity    hour_cos     0.276767
hour_cos    rush_hour    0.197467
temp        hour_sin     0.196330
atemp       hour_sin     0.189528
windspeed   hour_sin     0.182104
            hour_cos     0.181089
dtype: float64


## 공선성(상관) 점검 피드백

### 핵심 관찰 1) temp ↔ atemp 상관 0.985 (매우 높음)
- 둘을 동시에 쓰면 정보가 거의 중복됩니다.
- 트리 모델(XGB)에서는 성능이 망가지진 않지만,
  - 분할이 분산되고
  - time split 일반화에서 불필요한 복잡도가 늘 수 있어
  **정리 후보 1순위**입니다.

### 핵심 관찰 2) rush_hour ↔ rush_work 상관 0.785 (높음)
- rush_work는 rush_hour에 workingday를 곱한 파생인데,
  rush_hour 자체와 강하게 겹칩니다.
- 둘 다 넣을 필요가 없을 수 있으니 **둘 중 하나만 남기는 실험 가치**가 있습니다.

### 보조 관찰) temp/atemp ↔ month_sin/cos 상관(0.5~0.69)
- 계절성(월)과 기온이 당연히 연동되면서 생긴 상관입니다.
- 이건 중복이라기보다 “현상 자체”라서 섣불리 month_sin/cos를 빼기보단,
  temp/atemp 정리부터가 우선입니다.

In [49]:
# atemp 제거: X_train4_ts/X_valid4_ts에서 컬럼 제거
X_train_drop = X_train4_ts.drop(columns=["atemp"])
X_valid_drop = X_valid4_ts.drop(columns=["atemp"])

# numeric_cols4에서도 atemp 제거한 버전 생성
numeric_cols_drop = [c for c in numeric_cols4 if c != "atemp"]

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess_drop_atemp = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols_drop),
    ],
    remainder="drop"
)

fit_eval_xgb(preprocess_drop_atemp, X_train_drop, X_valid_drop, label="XGB time split - drop atemp")


[XGB time split - drop atemp]
RMSLE: 0.323, RMSE: 69.549, MAE: 45.841


## atemp 제거 결과 피드백 (time split)

### 성능 비교 (time split 기준)
- 기존 best(규제강화 + mcw=10, atemp 포함): **RMSLE 0.329 / RMSE 69.169 / MAE 45.472**
- ✅ atemp 제거: **RMSLE 0.323 / RMSE 69.549 / MAE 45.841**

### 해석
- **RMSLE가 0.329 → 0.323으로 확실히 개선**되었습니다.  
  → 공선성(중복 정보) 제거가 **미래 구간 일반화**에 도움이 된 케이스로 보입니다.
- 다만 RMSE/MAE는 약간 악화(미세)했지만, 목표가 RMSLE 중심이라면 **충분히 이득**입니다.

### 결론
- 현재 목적(미래 구간 RMSLE 개선) 관점에서 **`atemp`는 제거하는 편이 더 낫습니다.**
- 즉, “중요도/공선성 기반 피처 정리”가 실제로 성능을 올리는 게 확인됐습니다.

# 모델 블렌딩과 스태킹

옵션 A) “2-모델 스태킹” (가장 추천, 간단/효율)

1층(base): XGB + LGBM

2층(meta): Ridge(또는 LinearRegression) 를 log1p(count) 공간에서 학습

검증: TimeSeriesSplit으로 OOF 생성 → 마지막 구간에서 최종 평가

옵션 B) “가중 평균(Weighted Blending)” (스태킹보다 더 안전/간단)

스태킹 대신 pred = w*pred_xgb + (1-w)*pred_lgbm

time split에서 w만 3~5개 후보로 비교

실전에서 스태킹보다 손이 덜 가고, 개선이 비슷하게 나오는 케이스도 많음

In [50]:
# atemp 제거 버전(time split 데이터/전처리)으로 블렌딩(가중 평균)
import numpy as np
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# 1) XGB (time split에 맞춘 규제 강한 설정 + mcw=10)
xgb = XGBRegressor(
    n_estimators=4000,
    learning_rate=0.02,
    max_depth=6,
    min_child_weight=10,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    reg_alpha=0.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

pipe_xgb = Pipeline([("prep", preprocess_drop_atemp), ("model", xgb)])
pipe_xgb.fit(X_train_drop, np.log1p(y_train))
pred_xgb = np.maximum(0, np.expm1(pipe_xgb.predict(X_valid_drop)))

# 2) LGBM (비슷한 규모로, 과적합 줄이도록 샘플링/리프 제한 약간)
lgbm = LGBMRegressor(
    n_estimators=8000,
    learning_rate=0.01,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

pipe_lgbm = Pipeline([("prep", preprocess_drop_atemp), ("model", lgbm)])
pipe_lgbm.fit(X_train_drop, np.log1p(y_train))
pred_lgbm = np.maximum(0, np.expm1(pipe_lgbm.predict(X_valid_drop)))

# 3) 블렌딩 (우선 0.5:0.5)
w = 0.5
pred_blend = w * pred_xgb + (1 - w) * pred_lgbm

print(f"[Blend XGB:{w:.1f} + LGBM:{1-w:.1f}]")
evaluate_regr(y_valid, pred_blend)

c:\Users\Admin\miniconda3\envs\ml_edu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[Blend XGB:0.5 + LGBM:0.5]
RMSLE: 0.323, RMSE: 68.019, MAE: 44.907


## 블렌딩(0.5/0.5) 결과 피드백 (atemp 제거 + time split)

### 결과
- 블렌딩: **RMSLE 0.323 / RMSE 68.019 / MAE 44.907**

### 해석
- RMSLE가 **0.323으로 유지**되었습니다.
  - 즉, (현재 best로 알고 있는) XGB 단일 모델의 RMSLE **0.323**과 **동급**입니다.
- 대신 RMSE/MAE가 꽤 좋아 보입니다.  
  → 블렌딩이 “큰 값에서의 절대 오차”를 줄여주는 방향으로 작동했을 가능성이 있습니다.

### 결론
- **RMSLE만 보면 개선은 미미/없음(동점)**  
- 하지만 **RMSE/MAE까지 고려하면 블렌딩은 가치가 있음**  
- 다음은 “가중치(w)”를 움직여서 RMSLE를 더 내릴 여지가 있는지 확인하는 단계가 맞습니다.

In [51]:
#가중치(w) 조정 실험
for w in [0.3, 0.5, 0.7]:
    pred_blend = w * pred_xgb + (1 - w) * pred_lgbm
    print(f"\n[Blend XGB:{w:.1f} + LGBM:{1-w:.1f}]")
    evaluate_regr(y_valid, pred_blend)


[Blend XGB:0.3 + LGBM:0.7]
RMSLE: 0.325, RMSE: 67.760, MAE: 44.714

[Blend XGB:0.5 + LGBM:0.5]
RMSLE: 0.323, RMSE: 68.019, MAE: 44.907

[Blend XGB:0.7 + LGBM:0.3]
RMSLE: 0.322, RMSE: 68.482, MAE: 45.210


## 블렌딩 가중치 비교 피드백 (atemp 제거 + time split)

### 결과 요약
- w=0.3 (XGB 0.3 / LGBM 0.7): RMSLE **0.325**, RMSE 67.760, MAE 44.714
- w=0.5 (XGB 0.5 / LGBM 0.5): RMSLE **0.323**, RMSE 68.019, MAE 44.907
- ✅ w=0.7 (XGB 0.7 / LGBM 0.3): RMSLE **0.322**, RMSE 68.482, MAE 45.210

### 결론 (목표가 RMSLE라면)
- **최적은 w=0.7 (XGB 70% + LGBM 30%)**
- RMSLE가 **0.323 → 0.322로 소폭 개선**되었습니다.
- RMSE/MAE는 w가 작을수록(=LGBM 비중↑) 좋아지는 경향이 있어,
  “절대 오차”까지 챙기면 선택이 달라질 수 있습니다.

### 추천
- **RMSLE 최우선(캐글)**: ✅ `pred = 0.7*XGB + 0.3*LGBM` 채택
- **RMSE/MAE도 중요**: w=0.3이 더 유리하지만 RMSLE가 올라가므로 목적에 따라 선택

스태킹의 기대값이 낮아 보입니다.
이미 **블렌딩(0.7/0.3)**에서 RMSLE가 0.322까지 내려왔고, 이 정도면 두 모델의 예측이 꽤 잘 결합된 상태라서, 스태킹이 추가로 가져올 이득이 **아주 미세(또는 0)**일 가능성이 큽니다.

OOF는 Out-Of-Fold의 약자

교차검증(CV)에서 “내가 학습에 쓰지 않은 폴드(검증 폴드)”에 대해 만든 예측값

In [52]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# 전제(이미 존재해야 함):
# - X_train_drop, X_valid_drop : atemp 제거된 time split 데이터
# - y_train, y_valid : time split 타깃
# - preprocess_drop_atemp : atemp 제거 버전 전처리기
# - evaluate_regr : RMSLE/RMSE/MAE 출력 함수

# 1) Base 모델(시간기반에서 성능 좋았던 설정)
base_xgb = XGBRegressor(
    n_estimators=4000,
    learning_rate=0.02,
    max_depth=6,
    min_child_weight=10,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    reg_alpha=0.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

base_lgbm = LGBMRegressor(
    n_estimators=8000,
    learning_rate=0.01,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# 2) TimeSeriesSplit로 OOF 예측 생성 (train 구간 내부에서만)
tscv = TimeSeriesSplit(n_splits=3)

oof_xgb = np.zeros(len(X_train_drop))
oof_lgbm = np.zeros(len(X_train_drop))

Xtr = X_train_drop.reset_index(drop=True)
ytr = y_train.reset_index(drop=True)

for fold, (tr_idx, va_idx) in enumerate(tscv.split(Xtr), 1):
    # XGB
    pipe_xgb = Pipeline([("prep", preprocess_drop_atemp), ("model", base_xgb)])
    pipe_xgb.fit(Xtr.iloc[tr_idx], np.log1p(ytr.iloc[tr_idx]))
    pred_va_xgb = np.expm1(pipe_xgb.predict(Xtr.iloc[va_idx]))
    oof_xgb[va_idx] = np.maximum(0, pred_va_xgb)

    # LGBM
    pipe_lgbm = Pipeline([("prep", preprocess_drop_atemp), ("model", base_lgbm)])
    pipe_lgbm.fit(Xtr.iloc[tr_idx], np.log1p(ytr.iloc[tr_idx]))
    pred_va_lgbm = np.expm1(pipe_lgbm.predict(Xtr.iloc[va_idx]))
    oof_lgbm[va_idx] = np.maximum(0, pred_va_lgbm)

# 3) Meta 모델(Ridge): OOF 예측 2개를 입력으로 log1p(y)를 학습
meta_X = np.column_stack([oof_xgb, oof_lgbm])
meta_y = np.log1p(ytr.values)

meta = Ridge(alpha=1.0, random_state=42)
meta.fit(np.log1p(meta_X + 1e-9), meta_y)  # 안정성을 위해 log1p(예측) 공간에서 학습

# 4) 외부 valid(time split valid)에 대해 base 예측 → meta 예측
pipe_xgb_full = Pipeline([("prep", preprocess_drop_atemp), ("model", base_xgb)])
pipe_xgb_full.fit(X_train_drop, np.log1p(y_train))
pred_valid_xgb = np.maximum(0, np.expm1(pipe_xgb_full.predict(X_valid_drop)))

pipe_lgbm_full = Pipeline([("prep", preprocess_drop_atemp), ("model", base_lgbm)])
pipe_lgbm_full.fit(X_train_drop, np.log1p(y_train))
pred_valid_lgbm = np.maximum(0, np.expm1(pipe_lgbm_full.predict(X_valid_drop)))

valid_meta_X = np.column_stack([pred_valid_xgb, pred_valid_lgbm])
pred_valid_log = meta.predict(np.log1p(valid_meta_X + 1e-9))
pred_stack = np.maximum(0, np.expm1(pred_valid_log))

print("[Stacking: XGB+LGBM -> Ridge meta] (time split)")
evaluate_regr(y_valid, pred_stack)

r2 = r2_score(y_valid, pred_stack)
print(f"R2: {r2:.4f}")

c:\Users\Admin\miniconda3\envs\ml_edu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\Admin\miniconda3\envs\ml_edu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\Admin\miniconda3\envs\ml_edu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\Admin\miniconda3\envs\ml_edu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[Stacking: XGB+LGBM -> Ridge meta] (time split)
RMSLE: 0.868, RMSE: 168.423, MAE: 115.106
R2: 0.4009


## 결론: 현재 상태에서는 “스태킹은 불필요”가 아니라 “조건이 안 맞게 구현되어 실패”
- 스태킹 자체는 가능하지만,
- time series에서 OOF를 제대로 만들려면 “OOF가 100% 채워지는 설계”가 필요합니다.
- 반면 지금은 **블렌딩이 이미 0.322까지 개선**을 확인했고, 구현도 단순/안정적입니다.

✅ 따라서 현 단계 최선의 선택은 **스태킹 대신 블렌딩(0.7 XGB + 0.3 LGBM)** 유지입니다.

# 최종 (블렌딩) 모델의 성능 평가

In [53]:
from sklearn.metrics import r2_score
w = 0.7
pred_blend = w * pred_xgb + (1-w) * pred_lgbm
print("R2 (blend):", r2_score(y_valid, pred_blend))

R2 (blend): 0.9009516109436924


# 결과 제출

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# =========================
# 0) 경로 설정
# =========================
TRAIN_PATH = "data/bike_train.csv"
TEST_PATH  = "data/bike_test.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# =========================
# 1) Feature Engineering (train/test 동일)
#    - atemp 제거(공선성 정리)
#    - hour/month 주기(sin/cos)
#    - rush/weekend/rush_work
# =========================
def make_features(df):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])

    df["year"] = df["datetime"].dt.year
    df["month"] = df["datetime"].dt.month
    df["day"] = df["datetime"].dt.day
    df["hour"] = df["datetime"].dt.hour
    df["dayofweek"] = df["datetime"].dt.dayofweek

    # 주기 피처
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

    # rush/weekend
    df["rush_hour"] = df["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["rush_work"] = df["rush_hour"] * df["workingday"]

    return df

train_fe = make_features(train)
test_fe  = make_features(test)

# =========================
# 2) X/y 구성 (누수 제거 + atemp 제거)
# =========================
drop_cols = [c for c in ["datetime", "casual", "registered", "atemp"] if c in train_fe.columns]
X = train_fe.drop(columns=["count"] + drop_cols)
y = train_fe["count"].copy()

X_test = test_fe.drop(columns=[c for c in ["datetime", "casual", "registered", "atemp"] if c in test_fe.columns])

# =========================
# 3) 전처리 (범주형 원-핫 + 수치형 스케일)
# =========================
categorical_cols = ["season","holiday","workingday","weather","year","month","day","hour","dayofweek"]
numeric_cols = ["temp","humidity","windspeed",
                "hour_sin","hour_cos","month_sin","month_cos",
                "rush_hour","is_weekend","rush_work"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols),
    ],
    remainder="drop"
)

# =========================
# 4) 모델 정의 (log1p 학습)
# =========================
xgb = XGBRegressor(
    n_estimators=4000,
    learning_rate=0.02,
    max_depth=6,
    min_child_weight=10,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    reg_alpha=0.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

lgbm = LGBMRegressor(
    n_estimators=8000,
    learning_rate=0.01,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

pipe_xgb  = Pipeline([("prep", preprocess), ("model", xgb)])
pipe_lgbm = Pipeline([("prep", preprocess), ("model", lgbm)])

# =========================
# 5) 전체 train 학습 → test 예측 → 블렌딩
# =========================
y_log = np.log1p(y)

pipe_xgb.fit(X, y_log)
pred_xgb = np.maximum(0, np.expm1(pipe_xgb.predict(X_test)))

pipe_lgbm.fit(X, y_log)
pred_lgbm = np.maximum(0, np.expm1(pipe_lgbm.predict(X_test)))

w = 0.7  # time split에서 RMSLE가 가장 좋았던 가중치
pred = w * pred_xgb + (1 - w) * pred_lgbm
pred = np.maximum(0, pred)

# =========================
# 6) 제출 파일 생성
# =========================
submission = pd.DataFrame({
    "datetime": test["datetime"],  # 원본 문자열/형식 유지
    "count": pred
})
submission.to_csv("submission.csv", index=False)

print("saved: submission.csv")
display(submission.head())

c:\Users\Admin\miniconda3\envs\ml_edu\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


saved: submission.csv


,datetime,count
0,2011-01-20 00:00:00,11.803984
1,2011-01-20 01:00:00,3.729891
2,2011-01-20 02:00:00,2.814695
3,2011-01-20 03:00:00,2.378145
4,2011-01-20 04:00:00,1.379102
